# Music Annotation Project V8.8F
## Notebook 57R1: Registered-Replay Release Control and Notebook 09 Disposition

**Runtime:** CPU. Select **Runtime → Run all**.

This notebook closes the V8.8 registered-replay experiment. It consumes the exact completed Notebook 56 supervised-pilot run, validates its recorded Notebook 53R3 → Notebook 54 → Notebook 55 → Notebook 56 lineage, packages the approved page outputs, records the intended role of Notebook 09, and optionally activates a release pointer.

This notebook does **not**:

- create automatic annotation-mask authority;
- retune the registered replay or observed-print decision;
- reinterpret Notebook 09 `final/` pages as the controlled comparator;
- require Notebook 09 and V8.8 to agree outside M3;
- silently activate a release;
- retire Notebook 09 as comparator, fallback or exception route.

The first run should remain a **dry run** with `AUTHORISE_RELEASE = False`. Inspect the candidate record and upload package. Activation requires exact run overrides, an approver, an approval reference and an explicit candidate identity.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q pandas numpy opencv-python-headless pillow


## 0. Configuration

For a definitive decision, paste the exact completed Notebook 56 path into `NOTEBOOK56_RUN_OVERRIDE`.

Leave `AUTHORISE_RELEASE = False` for the first run. The dry run writes a complete candidate and a stable `release_candidate_id`. To activate later, rerun with the same exact Notebook 56 override, paste the dry-run candidate ID into `EXPECTED_RELEASE_CANDIDATE_ID`, and supply the approval fields.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import platform
import shutil
import sys
import zipfile

import cv2
import numpy as np
import pandas as pd

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')
N56_ROOT = ROOT / 'm5_remove_supervised_pilot_v8_8e'
RELEASE_ROOT = ROOT / 'v8_8_registered_replay_release_control'
ACTIVE_RELEASE_POINTER = RELEASE_ROOT / 'active_release.json'

# Set this to the exact completed Notebook 56 run for the definitive decision.
NOTEBOOK56_RUN_OVERRIDE = None

# Release decision. This does not enable annotated-only automatic operation.
REQUESTED_DISPOSITION = 'PRIMARY_CORRECTION_ENGINE_WITH_SUPERVISED_APPROVAL'
NOTEBOOK09_DISPOSITION = 'RETAINED_COMPARATOR_FALLBACK_EXCEPTION_REVIEWER'
MASK_AUTHORITY_MODE = 'AUTHORITATIVE_REVIEWED_MASK_REQUIRED'
AUTOMATIC_ANNOTATED_ONLY_ENABLED = False

# Optional scoped release. Leave None for the full Notebook 56 roster.
RELEASE_PAGE_ALLOWLIST = None
RELEASE_SOURCE_GROUP_ALLOWLIST = None

# Dry-run by default.
AUTHORISE_RELEASE = False
EXPECTED_RELEASE_CANDIDATE_ID = None
APPROVER = None
APPROVAL_REFERENCE = None
RELEASE_NOTES = (
    'V8.8 registered replay is preferred for structural preservation, '
    'subject to supervised approval and authoritative reviewed masks. '
    'Notebook 09 remains available for comparator, fallback and exception review.'
)

# Findings intentionally deferred to a later separately registered experiment.
NEXT_EXPERIMENT_FINDINGS = [
    {
        'finding_id': 'COSMETIC_RESIDUAL_ANNOTATION',
        'description': (
            'V8.8 can leave faint residual annotation or shadowing, especially '
            'on cleaner white-background regions where Notebook 09 can appear cleaner.'
        ),
        'classification': 'COSMETIC_QUALITY',
        'v8_8_change_authorised': False,
        'future_requirement': (
            'Investigate selective low-contrast residue cleanup without weakening '
            'preservation of genuine printed notation.'
        ),
    },
    {
        'finding_id': 'LIGHT_OBSERVED_PRINT_PRESERVATION',
        'description': (
            'Where Notebook 09 removes genuine print to white, V8.8 can preserve '
            'the genuine observed feature as light or grey because it copies the '
            'observed byte rather than synthesising black print.'
        ),
        'classification': 'EXPECTED_EXACT_OBSERVED_BYTE_BEHAVIOUR',
        'v8_8_change_authorised': False,
        'future_requirement': (
            'Any darkening or enhancement must be a separate non-generative or '
            'restoration hypothesis with its own controls and fresh validation.'
        ),
    },
    {
        'finding_id': 'DENSE_REGION_STRENGTH',
        'description': (
            'V8.8 was materially preferable in dense or visually contaminated areas '
            'because it preserved more genuine printed information.'
        ),
        'classification': 'REGRESSION_REQUIREMENT',
        'v8_8_change_authorised': False,
        'future_requirement': (
            'Future challengers must preserve this dense-region advantage and must '
            'not trade it away for cosmetic whitening.'
        ),
    },
]

EXPECTED_N56_VERDICTS = {'PASS_SUPERVISED_PILOT_COMPLETE'}
ALLOWED_APPROVED_SOURCES = {
    'V8.8_registered_replay',
    'Notebook09_proposal_fallback',
}
ALLOWED_DECISIONS = {
    'accept_v88',
    'retain_notebook09',
    'mask_review_required',
}

RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S') + '_notebook57R1_v8_8f'
OUT = RELEASE_ROOT / 'runs' / RUN_ID
CANDIDATE = OUT / 'release_candidate'
TABLES = OUT / 'tables'
AUDIT = OUT / 'audit'
UPLOAD = OUT / 'upload'
FINAL = CANDIDATE / 'final'

for path in [OUT, CANDIDATE, TABLES, AUDIT, UPLOAD, FINAL]:
    path.mkdir(parents=True, exist_ok=False)

print('Notebook 57R1 output:', OUT)


## 1. Helpers and complete-run resolution


In [ ]:
def require(condition, message):
    if not bool(condition):
        raise RuntimeError(message)


def read_json(path):
    path = Path(path)
    require(path.exists(), f'Missing JSON file: {path}')
    value = json.loads(path.read_text())
    require(isinstance(value, dict), f'JSON root must be an object: {path}')
    return value


def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = Path(str(path) + '.tmp')
    temporary.write_text(json.dumps(value, indent=2, sort_keys=True, default=str))
    temporary.replace(path)


def sha256_file(path):
    path = Path(path)
    require(path.exists() and path.is_file(), f'Missing file for SHA-256: {path}')
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def object_hash(value):
    payload = json.dumps(value, sort_keys=True, separators=(',', ':'), default=str)
    return hashlib.sha256(payload.encode()).hexdigest()


def latest_complete(root, required):
    root = Path(root)
    hits = []
    if root.exists():
        for path in root.rglob('*'):
            if path.is_dir() and all((path / item).exists() for item in required):
                hits.append(path)
    return max(hits, key=lambda path: path.stat().st_mtime) if hits else None


def resolve_recorded_path(value, label):
    require(value is not None and str(value).strip(), f'Missing recorded path: {label}')
    path = Path(str(value))
    require(path.exists(), f'Recorded {label} does not exist: {path}')
    return path


def normalise_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if pd.isna(value):
        return None
    return str(value).strip().lower() in {'true', '1', 'yes', 'y'}


def directory_inventory(root):
    rows = []
    root = Path(root)
    for path in sorted(root.rglob('*')):
        if path.is_file():
            rows.append({
                'relative_path': str(path.relative_to(root)),
                'bytes': path.stat().st_size,
                'sha256': sha256_file(path),
            })
    return pd.DataFrame(rows)


## 2. Resolve Notebook 56 and the registered-replay lineage

Notebook 56 is authoritative for the exact upstream run paths. Notebook 57R1 does not independently choose the latest Notebook 53, 54, 55 or Notebook 09 run.


In [ ]:
N56 = (
    Path(NOTEBOOK56_RUN_OVERRIDE)
    if NOTEBOOK56_RUN_OVERRIDE
    else latest_complete(
        N56_ROOT,
        [
            'notebook56_verdict.json',
            'pilot_approval.json',
            'tables/pilot_page_decisions.csv',
            'tables/final_page_inventory.csv',
            'final',
        ],
    )
)
require(N56 is not None, 'No complete Notebook 56 run found.')

v56 = read_json(N56 / 'notebook56_verdict.json')
approval56 = read_json(N56 / 'pilot_approval.json')

require(
    v56.get('verdict') in EXPECTED_N56_VERDICTS,
    f'Notebook 56 verdict is not releasable: {v56.get("verdict")}',
)
require(v56.get('first_failed_gate') is None, 'Notebook 56 records a failed gate.')
require(v56.get('automatic_approval') is False, 'Notebook 56 unexpectedly records automatic approval.')
require(v56.get('automatic_output_authority') is False, 'Notebook 56 unexpectedly has automatic output authority.')
require(v56.get('promotion_authority') is False, 'Notebook 56 unexpectedly has promotion authority.')
require(
    v56.get('notebook09_comparison_source') == 'proposals',
    'Notebook 56 did not use Notebook 09 proposals as the controlled comparator.',
)

N55 = resolve_recorded_path(v56.get('notebook55_run'), 'Notebook 55 run')
N53 = resolve_recorded_path(v56.get('notebook53_run'), 'Notebook 53 run')
N09 = resolve_recorded_path(v56.get('notebook09_run'), 'Notebook 09 run')

v55 = read_json(N55 / 'notebook55_verdict.json')

# Notebook 54 may be recorded directly by Notebook 55 or supplied by its expanded provenance.
N54 = None
if v55.get('notebook54_run'):
    N54 = resolve_recorded_path(v55.get('notebook54_run'), 'Notebook 54 run')
elif v56.get('notebook54_run'):
    N54 = resolve_recorded_path(v56.get('notebook54_run'), 'Notebook 54 run')

require(
    v55.get('verdict') == 'PASS_REGISTERED_REPLAY_NOTEBOOK09_SHADOW_EVIDENCE_READY_FOR_SUPERVISED_PILOT',
    f'Unexpected Notebook 55 verdict: {v55.get("verdict")}',
)
require(v55.get('first_failed_gate') is None, 'Notebook 55 records a failed gate.')
require(v55.get('historical_registered_replay') is True, 'Notebook 55 is not registered-replay evidence.')
require(v55.get('historical_code2_exclusion_replayed') is True, 'Notebook 55 did not replay the historical code-2 exclusion.')
require(v55.get('safe_protect_operationally_authorised') is False, 'SAFE_PROTECT unexpectedly has operational authority.')
require(v55.get('automatic_output_authority') is False, 'Notebook 55 unexpectedly has automatic output authority.')

require((N09 / 'proposals').exists(), 'Recorded Notebook 09 run lacks proposals/.')

source_version = v56.get('notebook53_version', v55.get('notebook53_version'))
source_mode = v56.get('notebook53_mode', v55.get('notebook53_mode'))

lineage = {
    'notebook53_run': str(N53),
    'notebook54_run': str(N54) if N54 else None,
    'notebook55_run': str(N55),
    'notebook56_run': str(N56),
    'notebook09_run': str(N09),
    'notebook09_comparison_directory': str(N09 / 'proposals'),
    'notebook53_version': source_version,
    'notebook53_mode': source_mode,
    'historical_registered_replay': True,
    'historical_code2_exclusion_replayed': True,
}

atomic_json(AUDIT / 'resolved_lineage.json', lineage)

print(json.dumps(lineage, indent=2))


## 3. Validate Notebook 56 decisions and exact approved source identities

Notebook 57R1 verifies the source paths and hashes recorded by Notebook 56. It does not guess whether a Notebook 09 page should come from `proposals/` or `final/`.


In [ ]:
decisions = pd.read_csv(N56 / 'tables' / 'pilot_page_decisions.csv', dtype={'page': str})
final_inventory = pd.read_csv(N56 / 'tables' / 'final_page_inventory.csv', dtype={'page': str})

required_decision_columns = {
    'page',
    'decision',
    'approved_source',
    'final_path',
    'final_sha256',
    'notebook09_source_path',
    'v88_source_path',
}
missing_columns = sorted(required_decision_columns - set(decisions.columns))
require(not missing_columns, f'Notebook 56 decision table lacks columns: {missing_columns}')

require(not decisions.page.duplicated().any(), 'Notebook 56 decision table contains duplicate pages.')
require(not final_inventory.page.duplicated().any(), 'Notebook 56 final inventory contains duplicate pages.')
require(set(decisions.page) == set(final_inventory.page), 'Decision and final-page inventory rosters differ.')
require(set(decisions.approved_source).issubset(ALLOWED_APPROVED_SOURCES), f'Unexpected approved_source values: {sorted(set(decisions.approved_source) - ALLOWED_APPROVED_SOURCES)}')
require(set(decisions.decision).issubset(ALLOWED_DECISIONS), f'Unexpected decision values: {sorted(set(decisions.decision) - ALLOWED_DECISIONS)}')

if RELEASE_PAGE_ALLOWLIST is not None:
    requested = set(map(str, RELEASE_PAGE_ALLOWLIST))
    missing_requested = sorted(requested - set(decisions.page))
    require(not missing_requested, f'Release allowlist pages are absent: {missing_requested}')
    decisions = decisions[decisions.page.isin(requested)].copy()

if RELEASE_SOURCE_GROUP_ALLOWLIST is not None:
    require('source_group' in decisions.columns, 'Source-group release scope requested but decisions lack source_group.')
    requested_groups = set(map(str, RELEASE_SOURCE_GROUP_ALLOWLIST))
    decisions = decisions[decisions.source_group.astype(str).isin(requested_groups)].copy()

require(len(decisions) > 0, 'Release scope is empty.')

verification_rows = []
for row in decisions.itertuples(index=False):
    final_path = Path(row.final_path)
    require(final_path.exists(), f'{row.page}: Notebook 56 final page missing: {final_path}')
    require(sha256_file(final_path) == row.final_sha256, f'{row.page}: Notebook 56 final SHA-256 mismatch.')

    if row.approved_source == 'V8.8_registered_replay':
        source_path = Path(row.v88_source_path)
        expected_decision = 'accept_v88'
    else:
        source_path = Path(row.notebook09_source_path)
        expected_decision = row.decision
        require(
            str(source_path).startswith(str(N09 / 'proposals')),
            f'{row.page}: Notebook 09 fallback is not from the controlled proposals directory: {source_path}',
        )

    require(source_path.exists(), f'{row.page}: approved source missing: {source_path}')
    require(row.decision == expected_decision or row.approved_source == 'Notebook09_proposal_fallback', f'{row.page}: inconsistent decision/source.')

    final_image = cv2.imread(str(final_path), cv2.IMREAD_UNCHANGED)
    source_image = cv2.imread(str(source_path), cv2.IMREAD_UNCHANGED)
    require(final_image is not None and source_image is not None, f'{row.page}: unreadable final/source image.')
    require(final_image.shape == source_image.shape, f'{row.page}: final/source shape mismatch.')
    exact = bool(np.array_equal(final_image, source_image))
    require(exact, f'{row.page}: final page is not byte-equivalent after decoding to its approved source.')

    inventory_match = final_inventory[final_inventory.page == row.page]
    require(len(inventory_match) == 1, f'{row.page}: final inventory row missing or duplicated.')
    require(
        str(inventory_match.iloc[0].sha256) == sha256_file(final_path),
        f'{row.page}: final inventory SHA-256 mismatch.',
    )

    verification_rows.append({
        'page': row.page,
        'decision': row.decision,
        'approved_source': row.approved_source,
        'approved_source_path': str(source_path),
        'approved_source_sha256': sha256_file(source_path),
        'notebook56_final_path': str(final_path),
        'notebook56_final_sha256': sha256_file(final_path),
        'decoded_array_exact': exact,
    })

verification = pd.DataFrame(verification_rows).sort_values('page')
verification.to_csv(AUDIT / 'approved_source_verification.csv', index=False)

print(verification.approved_source.value_counts())
print('Verified pages:', len(verification))


## 4. Technical, pilot and release-control gates

Notebook 09 disagreement is evidence and exception burden, not a requirement for zero disagreement. The integrity gate is the registered replay's own immutability and exact-equivalence chain.


In [ ]:
# Read Notebook 54 only when its path is durably recorded.
v54 = None
if N54 is not None:
    candidates = [
        N54 / 'notebook54_verdict.json',
        N54 / 'v8_8c_verdict.json',
        N54 / 'strict_exact_equivalence_verdict.json',
    ]
    existing = [path for path in candidates if path.exists()]
    require(len(existing) == 1, f'Expected one recognised Notebook 54 verdict file, found: {existing}')
    v54 = read_json(existing[0])

approval_counts = approval56.get('counts', {})

technical_gates = {
    'notebook55_registered_replay_shadow_pass': v55.get('verdict') == 'PASS_REGISTERED_REPLAY_NOTEBOOK09_SHADOW_EVIDENCE_READY_FOR_SUPERVISED_PILOT',
    'notebook55_no_failed_gate': v55.get('first_failed_gate') is None,
    'historical_registered_replay_confirmed': v55.get('historical_registered_replay') is True,
    'historical_code2_exclusion_confirmed': v55.get('historical_code2_exclusion_replayed') is True,
    'safe_protect_not_operationally_authorised': v55.get('safe_protect_operationally_authorised') is False,
    'notebook55_no_automatic_output_authority': v55.get('automatic_output_authority') is False,
    'notebook09_controlled_source_is_proposals': v56.get('notebook09_comparison_source') == 'proposals',
    'approved_source_files_exact': bool(len(verification) and verification.decoded_array_exact.all()),
}

if v54 is not None:
    technical_gates['notebook54_no_failed_stage'] = v54.get('first_failed_stage') is None
    technical_gates['notebook54_pass_like_verdict'] = str(v54.get('verdict', '')).startswith('PASS')

pilot_gates = {
    'notebook56_pass': v56.get('verdict') in EXPECTED_N56_VERDICTS,
    'notebook56_no_failed_gate': v56.get('first_failed_gate') is None,
    'notebook56_human_approval_recorded': (N56 / 'pilot_approval.json').exists(),
    'notebook56_no_automatic_approval': v56.get('automatic_approval') is False,
    'notebook56_no_automatic_output_authority': v56.get('automatic_output_authority') is False,
    'notebook56_no_promotion_authority': v56.get('promotion_authority') is False,
    'no_unreviewed_pages': int(approval_counts.get('unreviewed', 0)) == 0,
    'no_deferred_pages': int(approval_counts.get('defer', 0)) == 0,
    'release_scope_nonempty': len(verification) > 0,
}

release_control_gates = {
    'reviewed_mask_authority_retained': MASK_AUTHORITY_MODE == 'AUTHORITATIVE_REVIEWED_MASK_REQUIRED',
    'automatic_annotated_only_disabled': AUTOMATIC_ANNOTATED_ONLY_ENABLED is False,
    'notebook09_retained': NOTEBOOK09_DISPOSITION == 'RETAINED_COMPARATOR_FALLBACK_EXCEPTION_REVIEWER',
    'disposition_supported': REQUESTED_DISPOSITION in {
        'PRIMARY_CORRECTION_ENGINE_WITH_SUPERVISED_APPROVAL',
        'SHADOW_EXCEPTION_CHALLENGER',
        'RETAIN_NOTEBOOK09_PRIMARY',
    },
    'future_flaws_not_patched_post_pilot': all(item['v8_8_change_authorised'] is False for item in NEXT_EXPERIMENT_FINDINGS),
}

all_gates = {
    **{f'technical__{k}': v for k, v in technical_gates.items()},
    **{f'pilot__{k}': v for k, v in pilot_gates.items()},
    **{f'release_control__{k}': v for k, v in release_control_gates.items()},
}
first_failed_gate = next((key for key, value in all_gates.items() if value is not True), None)

status = {
    'technical_chain_status': 'PASS' if all(technical_gates.values()) else 'FAIL',
    'pilot_evidence_status': 'PASS' if all(pilot_gates.values()) else 'FAIL',
    'release_control_status': 'PASS' if all(release_control_gates.values()) else 'FAIL',
    'first_failed_gate': first_failed_gate,
}

atomic_json(AUDIT / 'gate_results.json', {
    'technical_gates': technical_gates,
    'pilot_gates': pilot_gates,
    'release_control_gates': release_control_gates,
    'status': status,
})

print(json.dumps(status, indent=2))
if first_failed_gate:
    raise RuntimeError(f'Notebook 57R1 blocked at {first_failed_gate}')


## 5. Build the immutable release candidate

The candidate packages the Notebook 56 approved final pages. It also preserves the exact approved source identity for every page. This is a correction-engine candidate requiring reviewed masks and supervised approval, not an annotated-only automatic release.


In [ ]:
page_rows = []
for row in verification.itertuples(index=False):
    source_final = Path(row.notebook56_final_path)
    destination = FINAL / f'{row.page}.png'
    shutil.copy2(source_final, destination)
    require(sha256_file(destination) == row.notebook56_final_sha256, f'{row.page}: copied final hash mismatch.')
    page_rows.append({
        'page': row.page,
        'decision': row.decision,
        'approved_source': row.approved_source,
        'approved_source_path': row.approved_source_path,
        'approved_source_sha256': row.approved_source_sha256,
        'candidate_final_path': str(destination),
        'candidate_final_sha256': sha256_file(destination),
        'bytes': destination.stat().st_size,
    })

scope = pd.DataFrame(page_rows).sort_values('page')
scope.to_csv(TABLES / 'release_scope.csv', index=False)

source_counts = scope.approved_source.value_counts().to_dict()
release_payload = {
    'release_schema': 'music-annotation-v8.8f-registered-replay-release-v1',
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'requested_disposition': REQUESTED_DISPOSITION,
    'notebook09_disposition': NOTEBOOK09_DISPOSITION,
    'mask_authority_mode': MASK_AUTHORITY_MODE,
    'automatic_annotated_only_enabled': AUTOMATIC_ANNOTATED_ONLY_ENABLED,
    'automatic_output_authority': False,
    'human_approval_required': True,
    'lineage': lineage,
    'notebook56_verdict_sha256': sha256_file(N56 / 'notebook56_verdict.json'),
    'notebook56_pilot_approval_sha256': sha256_file(N56 / 'pilot_approval.json'),
    'notebook56_decisions_sha256': sha256_file(N56 / 'tables' / 'pilot_page_decisions.csv'),
    'release_scope_sha256': sha256_file(TABLES / 'release_scope.csv'),
    'pages': len(scope),
    'approved_source_counts': source_counts,
    'future_experiment_findings': NEXT_EXPERIMENT_FINDINGS,
    'release_notes': RELEASE_NOTES,
}

release_candidate_id = object_hash(release_payload)
release_payload['release_candidate_id'] = release_candidate_id

atomic_json(CANDIDATE / 'release_manifest.json', release_payload)
atomic_json(CANDIDATE / 'notebook09_disposition.json', {
    'release_candidate_id': release_candidate_id,
    'disposition': NOTEBOOK09_DISPOSITION,
    'roles': [
        'controlled comparator',
        'proposal fallback',
        'exception reviewer',
        'rollback path',
        'retained regression evidence',
    ],
    'deleted_or_retired': False,
    'controlled_comparator_directory': str(N09 / 'proposals'),
})
atomic_json(CANDIDATE / 'next_experiment_backlog.json', {
    'release_candidate_id': release_candidate_id,
    'v8_8_policy_is_frozen': True,
    'findings': NEXT_EXPERIMENT_FINDINGS,
})

candidate_inventory = directory_inventory(CANDIDATE)
candidate_inventory.to_csv(CANDIDATE / 'candidate_inventory.csv', index=False)

print('Release candidate ID:', release_candidate_id)
print('Candidate pages:', len(scope))
print('Approved source counts:', source_counts)


## 6. Dry-run decision and optional activation

Activation is intentionally separate from candidate creation. The activation run must use an exact Notebook 56 override and the expected candidate ID from the dry run.


In [ ]:
activation_gates = {
    'technical_chain_pass': status['technical_chain_status'] == 'PASS',
    'pilot_evidence_pass': status['pilot_evidence_status'] == 'PASS',
    'release_controls_pass': status['release_control_status'] == 'PASS',
    'exact_notebook56_override_supplied': NOTEBOOK56_RUN_OVERRIDE is not None,
    'expected_candidate_id_supplied': EXPECTED_RELEASE_CANDIDATE_ID is not None,
    'expected_candidate_id_matches': EXPECTED_RELEASE_CANDIDATE_ID == release_candidate_id,
    'approver_supplied': bool(APPROVER and str(APPROVER).strip()),
    'approval_reference_supplied': bool(APPROVAL_REFERENCE and str(APPROVAL_REFERENCE).strip()),
    'authoritative_reviewed_mask_required': MASK_AUTHORITY_MODE == 'AUTHORITATIVE_REVIEWED_MASK_REQUIRED',
    'automatic_annotated_only_disabled': AUTOMATIC_ANNOTATED_ONLY_ENABLED is False,
}

release_status = 'DRY_RUN_CANDIDATE_READY'
previous_release = None

if AUTHORISE_RELEASE:
    failed_activation_gate = next((key for key, value in activation_gates.items() if value is not True), None)
    require(failed_activation_gate is None, f'Release activation blocked at {failed_activation_gate}')

    if ACTIVE_RELEASE_POINTER.exists():
        previous_release = read_json(ACTIVE_RELEASE_POINTER)
        atomic_json(
            OUT / 'previous_active_release.json',
            previous_release,
        )

    active_record = {
        'activated_utc': datetime.now(timezone.utc).isoformat(),
        'release_candidate_id': release_candidate_id,
        'release_manifest': str(CANDIDATE / 'release_manifest.json'),
        'release_run': str(OUT),
        'requested_disposition': REQUESTED_DISPOSITION,
        'notebook09_disposition': NOTEBOOK09_DISPOSITION,
        'mask_authority_mode': MASK_AUTHORITY_MODE,
        'automatic_annotated_only_enabled': False,
        'human_approval_required': True,
        'approver': APPROVER,
        'approval_reference': APPROVAL_REFERENCE,
        'previous_release_candidate_id': previous_release.get('release_candidate_id') if previous_release else None,
        'rollback_pointer': previous_release,
    }
    atomic_json(ACTIVE_RELEASE_POINTER, active_record)
    release_status = 'ACTIVE_RELEASE_POINTER_WRITTEN'
else:
    failed_activation_gate = None

release_decision = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'release_status': release_status,
    'authorise_release': AUTHORISE_RELEASE,
    'release_candidate_id': release_candidate_id,
    'requested_disposition': REQUESTED_DISPOSITION,
    'notebook09_disposition': NOTEBOOK09_DISPOSITION,
    'mask_authority_mode': MASK_AUTHORITY_MODE,
    'automatic_annotated_only_enabled': False,
    'human_approval_required': True,
    'activation_gates': activation_gates,
    'failed_activation_gate': failed_activation_gate,
    'approver': APPROVER,
    'approval_reference': APPROVAL_REFERENCE,
    'next_experiment_findings_recorded': True,
}
atomic_json(OUT / 'release_decision.json', release_decision)

print(json.dumps(release_decision, indent=2))


## 7. Monitoring, rollback and release interpretation


In [ ]:
monitoring = {
    'release_candidate_id': release_candidate_id,
    'monitoring_required': True,
    'metrics': [
        'human accept V8.8 rate',
        'Notebook 09 fallback rate',
        'mask-review-required rate',
        'defer rate',
        'source-group concentration',
        'outside-authority integrity failures',
        'faint residual annotation observations',
        'genuine-print preservation regressions',
    ],
    'stop_conditions': [
        'any outside-authority pixel drift',
        'any registered-replay or artifact-identity failure',
        'systematic loss of genuine printed notation',
        'new source-group failure concentration',
        'review burden incompatible with supervised operation',
    ],
    'rollback': {
        'primary': 'Use Notebook 09 proposal path for affected pages.',
        'secondary': 'Restore the previous active release pointer when present.',
        'automatic_rollback': False,
        'human_decision_required': True,
    },
}
atomic_json(CANDIDATE / 'monitoring_and_rollback.json', monitoring)

interpretation = {
    'technical_chain_status': status['technical_chain_status'],
    'pilot_evidence_status': status['pilot_evidence_status'],
    'release_control_status': status['release_control_status'],
    'release_status': release_status,
    'experiment_complete': True,
    'v8_8_policy_frozen': True,
    'automatic_annotation_mask_authority_validated': False,
    'automatic_annotated_only_operation_authorised': False,
    'notebook09_retained': True,
    'future_experiment_required_for_known_limitations': True,
}
atomic_json(OUT / 'notebook57R1_verdict.json', interpretation)

print(json.dumps(interpretation, indent=2))


## 8. Final upload package


In [ ]:
summary_lines = [
    'NOTEBOOK 57R1 / V8.8F REGISTERED-REPLAY RELEASE CONTROL',
    '',
    f'Release status: {release_status}',
    f'Release candidate ID: {release_candidate_id}',
    f'Pages in scope: {len(scope)}',
    f'V8.8 registered-replay pages: {source_counts.get("V8.8_registered_replay", 0)}',
    f'Notebook 09 proposal fallback pages: {source_counts.get("Notebook09_proposal_fallback", 0)}',
    f'Requested disposition: {REQUESTED_DISPOSITION}',
    f'Notebook 09 disposition: {NOTEBOOK09_DISPOSITION}',
    f'Mask authority: {MASK_AUTHORITY_MODE}',
    'Automatic annotated-only operation: False',
    'Human approval required: True',
    '',
    'Recorded future experiment findings:',
    '- Faint cosmetic annotation/shadow residue can remain on cleaner white areas.',
    '- Genuine print preserved beyond Notebook 09 may remain light/grey because V8.8 copies the observed byte.',
    '- V8.8 dense/mucky-region preservation is a regression requirement for future challengers.',
    '',
    'Notebook 09 remains comparator, fallback, exception reviewer and rollback evidence.',
]
(OUT / 'executive_summary.txt').write_text(chr(10).join(summary_lines))

curated = [
    OUT / 'executive_summary.txt',
    OUT / 'notebook57R1_verdict.json',
    OUT / 'release_decision.json',
    AUDIT / 'resolved_lineage.json',
    AUDIT / 'approved_source_verification.csv',
    AUDIT / 'gate_results.json',
    TABLES / 'release_scope.csv',
    CANDIDATE / 'release_manifest.json',
    CANDIDATE / 'notebook09_disposition.json',
    CANDIDATE / 'next_experiment_backlog.json',
    CANDIDATE / 'monitoring_and_rollback.json',
    CANDIDATE / 'candidate_inventory.csv',
]

package_rows = []
for source in curated:
    require(source.exists(), f'Missing curated release artefact: {source}')
    destination = UPLOAD / source.name
    if destination.exists():
        destination = UPLOAD / f'{source.parent.name}__{source.name}'
    shutil.copy2(source, destination)
    package_rows.append({
        'file': destination.name,
        'source': str(source),
        'bytes': destination.stat().st_size,
        'sha256': sha256_file(destination),
    })

pd.DataFrame(package_rows).to_csv(UPLOAD / 'upload_manifest.csv', index=False)
(UPLOAD / 'README.txt').write_text(
    'Notebook 57R1 V8.8F registered-replay release-control package.' + chr(10) +
    'The full approved-page candidate remains in release_candidate/final/.' + chr(10) +
    'No automatic annotation-mask authority or annotated-only operation is authorised.' + chr(10)
)
(UPLOAD / 'upload_file_list.txt').write_text(
    chr(10).join(sorted(path.name for path in UPLOAD.iterdir() if path.is_file())) + chr(10)
)

zip_path = UPLOAD / 'notebook57R1_v8_8f_upload_bundle.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(UPLOAD.iterdir()):
        if path.is_file() and path != zip_path:
            archive.write(path, path.name)

require(zip_path.exists() and zip_path.stat().st_size > 0, 'Upload ZIP creation failed.')

print(chr(10).join(summary_lines))
print(chr(10) + 'Upload package:', zip_path)
